# Creating a Pytorch full training loop

This can be used in many ways such as:
- **Model Evaluation**: Always evaluate your model on multiple metrics, not just accuracy. Use the 🤗 Evaluate library for comprehensive evaluation.

- **Hyperparameter Tuning**: Consider using libraries like Optuna or Ray Tune for systematic hyperparameter optimization.

- **Model Monitoring**: Track training metrics, learning curves, and validation performance throughout training.

- **Model Sharing**: Once trained, share your model on the Hugging Face Hub to make it available to the community.

- **Efficiency**: For large models, consider techniques like gradient checkpointing, parameter-efficient fine-tuning (LoRA, AdaLoRA), or quantization methods.

The code below will achieve the same results as part 2 without using the **Trainer** class, instead a training loop with modern PyTorch will be used

First lets do some basic preprocessing

In [23]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("SetFit/mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["text1"], example["text2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Repo card metadata block was not found. Setting CardData to empty.


#### Postprocessing steps:
before defining the **dataloaders** object that will be used to iterate over the batches. we must:

#### 1.1 **process **tokenized_datasets** with steps that the original **Trainer** class did for us**.
- Remove the columns corresponding to values the model does not expect (like the *'text1'* and *'text2'* columns).
- Rename the column *'label'* to *'labels'* (because the model expects the argument to be named labels).
- Set the format of the datasets so they return PyTorch tensors instead of lists.

In [24]:
tokenized_datasets = tokenized_datasets.remove_columns(["text1", "text2", "idx",'label_text'])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

#### 1.2. Define the dataloaders
#### 1.3. Check For Mistakes in the data processing
- #### Since  since we are setting *shuffle=True* for the training dataloader, the shape of the data will be slightly different

In [25]:
from torch.utils.data import DataLoader

# Define the Dataloaders
train_dataloader = DataLoader(
    tokenized_datasets["train"], 
    shuffle=True, 
    batch_size=32, 
    collate_fn=data_collator,
    num_workers=8
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], 
    batch_size=32, 
    collate_fn=data_collator,
    num_workers=8
)

# Check batches For Mistakes in the data processing
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 76]),
 'token_type_ids': torch.Size([32, 76]),
 'attention_mask': torch.Size([32, 76])}

# Create The Model

#### 2.1 instantiate the model like before

In [26]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

#To make sure that everything will go smoothly during training, 
# we pass our batch to this model:
outputs = model(**batch)
print("\n",outputs.loss, outputs.logits.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 tensor(0.5901, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


#### 2.2. We are missing two things:
1. Optimizer
2. Learning Rate Scheduler

#### We need to use the same defaults from the *Trainer* so we use:
- **Optimizer** --> AdamW
    - AdamW differs from Adam. in AdamW, weight decay is separated from the gradient-based parameter updates, leading to better regularization.
- **Learning Rate Scheduler** --> linear decay from the maximum value (5e-5) to 0. 
    - We need the number of training steps we will take, (epochs x # of training batches/length of training dataloader). 


In [27]:
from transformers import get_scheduler
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps) #345 Training Steps

345


#### Optional: If there is access to a GPU, we can define a *device* we'll put our model and our batches on. the CPU is used as a default

In [28]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

device(type='cuda')

# Let's Train!
#### The loop below Goes through the steps:
#### Forward pass (compute loss)  → Backward pass (compute gradients) → Optimizer step (update parameters) → Scheduler step (update learning rate) → Zero gradients (clear gradients)

In [29]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 
            max_norm=1.0
            )

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/345 [00:00<?, ?it/s]

In [30]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8700980392156863, 'f1': 0.9068541300527241}

### Optional: **Use Accelerate Library**
#### Accelerator enables distributed training on multiple GPUs or TPUs. handles the complexity of distributed training, mixed precision, and device placement automatically.

#### Let's put all the steps together!

In [31]:
# Let's put it all together and speed it up!
from tqdm.auto import tqdm
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

accelerator = Accelerator()

# 1. Define the Dataloaders
train_dataloader = DataLoader(
    tokenized_datasets["train"], 
    shuffle=True, 
    batch_size=32, # Set to 32 for quickest computation
    collate_fn=data_collator,
    num_workers=0 #Set to 0 for quickest computation
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], 
    batch_size=32, 
    collate_fn=data_collator,
    num_workers=0
)

# 2. create the training and validation dataloaders
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

# 3. Create Optimizer and Learning Rate Scheduler
num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# 4. Run the model
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/345 [00:00<?, ?it/s]

In [32]:
metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8627450980392157, 'f1': 0.903448275862069}

### OOP! We still need to validate the data and get the following metrics:
- Training Loss
- Validation Loss
- Accuracy
- F1

### The code below runs through the accelerated model, and prints metrics for each epoch with a final table output at the end


In [ ]:
import torch

from accelerate import Accelerator
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    get_scheduler,
)

accelerator = Accelerator()

# 1. Define the Dataloaders
train_dataloader = DataLoader(
    tokenized_datasets["train"], 
    shuffle=True, 
    batch_size=32, 
    collate_fn=data_collator,
    num_workers=0
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], 
    batch_size=32, 
    collate_fn=data_collator,
    num_workers=0
)

# 2. create the training and validation dataloaders
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
)

optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader,
    eval_dataloader,
    model,
    optimizer,
)

# 3. Create Optimizer and Learning Rate Scheduler
num_epochs = 3
num_training_steps = num_epochs * len(train_dl)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(
    range(num_training_steps),
    disable=not accelerator.is_local_main_process,
)

# Store the metrics for every epoch.
history = []

for epoch in range(num_epochs):

    # =====================================================
    # Training
    # =====================================================
    model.train()
    training_losses = []

    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss

        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        # Associate the average batch loss with every sample
        # in the current batch.
        batch_size = batch["labels"].shape[0]
        batch_losses = loss.detach().repeat(batch_size)

        gathered_training_losses = accelerator.gather_for_metrics(
            batch_losses
        )

        training_losses.append(gathered_training_losses)

        progress_bar.update(1)

    training_losses = torch.cat(training_losses)
    training_loss = training_losses.mean().item()

    # =====================================================
    # Validation
    # =====================================================
    model.eval()

    validation_losses = []
    all_predictions = []
    all_labels = []

    for batch in eval_dl:
        with torch.no_grad():
            outputs = model(**batch)

        validation_batch_loss = outputs.loss
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1)
        labels = batch["labels"]

        batch_size = labels.shape[0]
        batch_losses = validation_batch_loss.detach().repeat(batch_size)

        # Gather validation data from all Accelerator processes.
        gathered_loss = accelerator.gather_for_metrics(batch_losses)
        gathered_predictions = accelerator.gather_for_metrics(
            predictions
        )
        gathered_labels = accelerator.gather_for_metrics(labels)

        validation_losses.append(gathered_loss)
        all_predictions.append(gathered_predictions)
        all_labels.append(gathered_labels)

    validation_losses = torch.cat(validation_losses)
    all_predictions = torch.cat(all_predictions)
    all_labels = torch.cat(all_labels)

    validation_loss = validation_losses.mean().item()

    # =====================================================
    # Accuracy
    # =====================================================
    accuracy = (
        (all_predictions == all_labels)
        .float()
        .mean()
        .item()
    )

    # =====================================================
    # Binary F1 score
    # Label 1 is treated as the positive class.
    # =====================================================
    true_positives = (
        (all_predictions == 1) & (all_labels == 1)
    ).sum().float()

    false_positives = (
        (all_predictions == 1) & (all_labels == 0)
    ).sum().float()

    false_negatives = (
        (all_predictions == 0) & (all_labels == 1)
    ).sum().float()

    precision = true_positives / (
        true_positives + false_positives + 1e-8
    )

    recall = true_positives / (
        true_positives + false_negatives + 1e-8
    )

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    ).item()

    # =====================================================
    # Save this epoch's metrics
    # =====================================================
    epoch_metrics = {
        "epoch": epoch + 1,
        "training_loss": training_loss,
        "validation_loss": validation_loss,
        "accuracy": accuracy,
        "f1": f1,
    }

    history.append(epoch_metrics)

    # Print after every epoch.
    accelerator.print(
        f"\n{'=' * 45}\n"
        f"Epoch {epoch + 1}/{num_epochs}\n"
        f"{'=' * 45}\n"
        f"Training Loss:   {training_loss:.4f}\n"
        f"Validation Loss: {validation_loss:.4f}\n"
        f"Accuracy:        {accuracy:.4f}\n"
        f"F1:              {f1:.4f}\n"
    )

accelerator.wait_for_everyone()

# =========================================================
# Print a final summary containing all epochs
# =========================================================
if accelerator.is_main_process:
    print("\nTraining Summary")
    print("=" * 76)
    print(
        f"{'Epoch':<8}"
        f"{'Training Loss':<18}"
        f"{'Validation Loss':<20}"
        f"{'Accuracy':<15}"
        f"{'F1':<15}"
    )
    print("-" * 76)

    for metrics in history:
        print(
            f"{metrics['epoch']:<8}"
            f"{metrics['training_loss']:<18.4f}"
            f"{metrics['validation_loss']:<20.4f}"
            f"{metrics['accuracy']:<15.4f}"
            f"{metrics['f1']:<15.4f}"
        )


Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'bert-base-uncased'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/345 [00:00<?, ?it/s]


Epoch 1/3
Training Loss:   0.6005
Validation Loss: 0.5120
Accuracy:        0.7672
F1:              0.8376


Epoch 2/3
Training Loss:   0.4319
Validation Loss: 0.4288
Accuracy:        0.8235
F1:              0.8800


Epoch 3/3
Training Loss:   0.2457
Validation Loss: 0.4729
Accuracy:        0.8284
F1:              0.8833


Training Summary
Epoch   Training Loss     Validation Loss     Accuracy       F1             
----------------------------------------------------------------------------
1       0.6005            0.5120              0.7672         0.8376         
2       0.4319            0.4288              0.8235         0.8800         
3       0.2457            0.4729              0.8284         0.8833         


## Training Summary

| Epoch | Training Loss | Validation Loss | Accuracy | F1 Score |
|---------|-------------:|----------------:|---------:|---------:|
| 1 | 0.6005 | 0.5120 | 76.72% | 83.76% |
| 2 | 0.4319 | 0.4288 | 82.35% | 88.00% |
| 3 | 0.2457 | 0.4729 | 82.84% | 88.33% |

### Observations

- Training loss decreased steadily from **0.6005** to **0.2457**, indicating the model successfully learned patterns from the training data.
- Validation loss improved between **Epoch 1** and **Epoch 2** (**0.5120 → 0.4288**) but increased at **Epoch 3** (**0.4729**), suggesting the beginning of mild overfitting.
- Validation accuracy increased from **76.72%** to **82.84%** across the three epochs.
- The F1 score improved from **0.8376** to **0.8833**, indicating better overall performance in identifying paraphrase pairs.
- Although Epoch 3 achieved the highest accuracy and F1 score, Epoch 2 produced the lowest validation loss, representing the best balance between learning and generalization.

### Conclusion

The fine-tuned BERT model demonstrated consistent improvements in classification performance over three training epochs. While predictive performance continued to improve through Epoch 3, the increase in validation loss suggests that additional training beyond this point may lead to overfitting. Epochs 2 and 3 represent the strongest model checkpoints, depending on whether validation loss or predictive performance is prioritized.